# Generative AI 006 — Prompts in LangChain

The prompts component is pure string and message handling, so — unlike a model
call — **every claim in this lesson can be checked without an API key.** That is
what this notebook does.

| Part | What we check |
|---|---|
| A | a dynamic prompt: **45** possible prompts, **45/45** keep the safety line |
| B | when a typo is caught: call time, construction, or **never** |
| C | a role-less history mislabels **1 of 3** messages, silently |
| D | the `ChatPromptTemplate` trap — no error, wrong prompt |
| E | `MessagesPlaceholder`: 3 pieces + 2 stored = 4 messages |
| F | `template \| model` — the one advantage with no workaround |

Needs `langchain-core` only. Install with `pip install langchain-core`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")          # version warnings; we read them in Part G

import langchain_core
print("langchain_core", langchain_core.__version__)

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import (SystemMessage, HumanMessage, AIMessage,
                                     convert_to_messages)
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.runnables import Runnable

## Part A — Static against dynamic

A **static** prompt is a text box: the user writes the whole sentence. A
**dynamic** prompt is a template with named holes, and the user only picks the
values that go in them.

The argument for the second one is countable, so let's count it.

In [ ]:
import itertools

PAPERS  = ["Attention Is All You Need", "BERT", "GPT-3", "Diffusion Models", "AlphaFold"]
STYLES  = ["beginner-friendly", "technical", "code-oriented"]
LENGTHS = ["short", "medium", "long"]

template = PromptTemplate(
    template=(
        "Summarize the research paper titled {paper_input} with "
        "{style_input} explanation style and {length_input} length. "
        "If the information is missing, say 'Insufficient information "
        "available' rather than guessing."
    ),
    input_variables=["paper_input", "style_input", "length_input"],
    validate_template=True,
)

prompts = [
    template.invoke({"paper_input": p, "style_input": s, "length_input": l}).text
    for p, s, l in itertools.product(PAPERS, STYLES, LENGTHS)
]

print("possible prompts:", len(prompts))
print("that keep the anti-hallucination line:",
      sum("Insufficient information available" in t for t in prompts), "/", len(prompts))
print()
print(prompts[0])

assert len(prompts) == 45
assert all("Insufficient information available" in t for t in prompts)

45 prompts, every one of which you could print out and read before shipping. A
free text box can produce any string a user can type — that is not a big number,
it is an unbounded one, and none of it has been tested.

**The user picks the values. You keep the sentence.**

## Part B — When is a typo caught?

Write `{style}` in the template but declare `styl` in the variable list. Three
ways to build the prompt, three different answers — and one of them is not the
answer usually given.

In [ ]:
TEMPLATE = "Summarize {paper} in {style} style"

# (a) A plain f-string / str.format.
try:
    TEMPLATE.format(paper="X")
    print("(a) str.format: no error")
except KeyError as e:
    print("(a) str.format ->", type(e).__name__, e, "  [at CALL TIME]")

In [ ]:
# (b) validate_template=True
try:
    PromptTemplate(template=TEMPLATE, input_variables=["paper", "styl"],
                   validate_template=True)
    print("(b) no error")
except Exception as e:
    print("(b) validate_template=True ->", type(e).__name__, "  [at CONSTRUCTION]")
    print("    i.e. when the module is imported, on your machine")

In [ ]:
# (c) validation left off - the surprising one
t = PromptTemplate(template=TEMPLATE, input_variables=["paper", "styl"])
print("(c) constructed with no complaint")
print("    declared :", ["paper", "styl"])
print("    actual   :", t.input_variables)

assert t.input_variables == ["paper", "style"]
print()
print("LangChain threw away the declaration and re-derived the names from the")
print("template text, silently CORRECTING the typo. So the common claim that")
print("'PromptTemplate validates and f-strings do not' is too generous.")
print("Only validate_template=True finds the mistake early - and it is not")
print("the default, so you have to ask for it.")

| How you build it | What happens | When |
|---|---|---|
| `str.format` | `KeyError` | call time, every time |
| `validate_template=True` | `ValidationError` | **construction** |
| `PromptTemplate(...)` default | declaration ignored, typo corrected | **never** |

## Part C — What a role-less chat history costs

Now the chatbot half of the lesson. The naive fix for statelessness is to keep a
list of everything said and send the whole list each time. It works — and it
introduces a second bug that never announces itself.

In [ ]:
model = FakeListChatModel(responses=["2"])     # a stub; cycles, so always "2"

history = ["Which is greater, 2 or 0?"]
reply = model.invoke(history)
history.append(reply.content)                  # the AI's answer, as a bare string
history.append("Multiply the bigger number by 10")

print("the history the naive chatbot builds:")
for h in history:
    print("   ", repr(h))

In [ ]:
coerced = convert_to_messages(history)
typed = [HumanMessage(content="Which is greater, 2 or 0?"),
         AIMessage(content="2"),
         HumanMessage(content="Multiply the bigger number by 10")]

print(f"{'what the library makes of it':<32}{'what it should have been'}")
for c, t in zip(coerced, typed):
    print(f"   {type(c).__name__:<29}{type(t).__name__}")

wrong = sum(type(a).__name__ != type(b).__name__ for a, b in zip(coerced, typed))
print(f"\nmessages: {len(history)}    mislabelled: {wrong}")

assert wrong == 1

Every plain string becomes a `HumanMessage` — **including the model's own
reply**. The model is being told that the *user* said "2". It never sees that it
answered anything.

And notice: **no error was raised.** The call succeeds, you are billed, and the
answer is simply worse. This is not untidiness — the transcript you hand the
model is factually wrong.

In [ ]:
# The fix: say who said what.
messages = [SystemMessage(content="You are a helpful assistant")]
messages.append(HumanMessage(content="Which is greater, 2 or 0?"))
result = model.invoke(messages)
messages.append(AIMessage(content=result.content))      # labelled as the AI
messages.append(HumanMessage(content="Multiply the bigger number by 10"))

for m in messages:
    print(f"   {type(m).__name__:<14} | {m.content}")

# SystemMessage - an instruction, sent once at the start
# HumanMessage  - what the user said
# AIMessage     - what the model said
assert [type(m).__name__ for m in messages] == [
    "SystemMessage", "HumanMessage", "AIMessage", "HumanMessage"]

## Part D — The `ChatPromptTemplate` trap

`ChatPromptTemplate` makes a *list* of messages dynamic. The natural way to write
it — using the message classes you just learned — does not work, and does not
complain.

In [ ]:
values = {"domain": "cricket", "topic": "a googly"}

good = ChatPromptTemplate([
    ("system", "You are a helpful {domain} expert"),
    ("human", "Explain in simple terms what is {topic}"),
])
print("TUPLES   input_variables:", good.input_variables)
for m in good.invoke(values).messages:
    print(f"   {type(m).__name__:<14} | {m.content}")

In [ ]:
bad = ChatPromptTemplate([
    SystemMessage(content="You are a helpful {domain} expert"),
    HumanMessage(content="Explain in simple terms what is {topic}"),
])
print("OBJECTS  input_variables:", bad.input_variables, " <- empty!")
for m in bad.invoke(values).messages:
    print(f"   {type(m).__name__:<14} | {m.content}")

print("\nNo exception. The literal characters '{domain}' go to the model,")
print("you pay for the call, and the answer is nonsense.")

assert good.input_variables == ["domain", "topic"]
assert bad.input_variables == []
assert sum("{" in m.content for m in bad.invoke(values).messages) == 2

In [ ]:
# Does from_messages rescue you? No.
worse = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful {domain} expert"),
    HumanMessage(content="Explain what is {topic}"),
])
print("from_messages + objects  input_variables:", worse.input_variables)
assert worse.input_variables == []

print("\nRULE: build a ChatPromptTemplate from TUPLES.")
print("Use the message classes for actual messages, not template pieces.")

## Part E — `MessagesPlaceholder`

A customer asks for a refund on Monday and is told 3–5 days. On Wednesday they
ask "Where is my refund?" To answer, the model needs Monday's conversation —
which lives in a database, not in this session.

`MessagesPlaceholder` is a slot that holds a **whole list** of messages.

In [ ]:
template = ChatPromptTemplate([
    ("system", "You are a helpful customer support agent"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{query}"),
])

chat_history = [                                  # loaded from a database
    HumanMessage(content="I want a refund for order 4521"),
    AIMessage(content="Your refund is initiated. It takes 3-5 business days."),
]

prompt = template.invoke({"chat_history": chat_history, "query": "Where is my refund?"})

print(len(prompt.messages), "messages  (3 template pieces + 2 stored)")
for i, m in enumerate(prompt.messages, 1):
    print(f"  {i}. {type(m).__name__:<14} | {m.content}")

assert len(prompt.messages) == 4
assert [type(m).__name__ for m in prompt.messages] == [
    "SystemMessage", "HumanMessage", "AIMessage", "HumanMessage"]

In [ ]:
# And with no history it just collapses - so ONE template serves both a
# brand-new customer and a returning one.
empty = template.invoke({"chat_history": [], "query": "Hello"})
print(len(empty.messages), "messages when the history is empty")
assert len(empty.messages) == 2

## Part F — Why a template beats an f-string

Validation and save/load are conveniences. This one is structural.

In [ ]:
tpl = PromptTemplate.from_template("Explain {topic} in simple terms")
model = FakeListChatModel(responses=["A chain joins steps together."])

# Two calls, carrying the value between them:
prompt = tpl.invoke({"topic": "chains"})
print(model.invoke(prompt).content)

# One call:
chain = tpl | model
print(chain.invoke({"topic": "chains"}).content)

print("\nisinstance(tpl, Runnable):", isinstance(tpl, Runnable))
print("type of the chain        :", type(chain).__name__)

assert isinstance(tpl, Runnable)
assert type(chain).__name__ == "RunnableSequence"

An f-string cannot appear in that pipeline. It is a string: no `.invoke`,
nothing to pipe. **That is the advantage with no workaround**, and everything
later in this course is built by joining `Runnable`s like this.

## Part G — A version note worth reading

Most tutorials teach `template.save("template.json")` and
`load_prompt("template.json")` for reusing prompts. Check what your installed
version actually thinks of that.

In [ ]:
import warnings as _w

with _w.catch_warnings(record=True) as caught:
    _w.simplefilter("always")
    import tempfile, os
    from langchain_core.prompts import load_prompt
    t = PromptTemplate(
        template="Summarize the research paper titled {paper_input}",
        input_variables=["paper_input"])   # same template as the lesson
    path = os.path.join(tempfile.mkdtemp(), "template.json")
    t.save(path)
    load_prompt(path)

for w in caught:
    print(f"{w.category.__name__}: {str(w.message)[:110]}")

In [ ]:
# The current route:
from langchain_core.load import dumps, loads

blob = dumps(t)
restored = loads(blob)
print("serialised to", len(blob), "characters")
print("round-trips to", type(restored).__name__)
print(restored.invoke({"paper_input": "Attention Is All You Need"}).text)

assert len(blob) == 262          # the figure quoted in the lesson

print("\nNOTE: loads() carries a beta warning of its own, so the replacement")
print("is not settled either. Check your installed version rather than")
print("trusting any tutorial - including this one.")

## What to take away

- **A dynamic prompt bounds the problem.** 45 possible prompts, all testable,
  and **45/45** keep the instruction that stops the model guessing.
- **Validation is opt-in.** With it off, LangChain silently *corrects* a typo'd
  variable list. Only `validate_template=True` catches it, at construction.
- **A history of plain strings loses the speaker** — every entry becomes a
  `HumanMessage`, including the model's own reply. **1 of 3** mislabelled, no
  error raised.
- **The three message types** are `SystemMessage`, `HumanMessage`, `AIMessage`.
- **Build `ChatPromptTemplate` from tuples.** Message objects leave
  `input_variables` empty and send `{domain}` literally. `from_messages` does not
  help.
- **`MessagesPlaceholder`** expands a stored conversation in place — 3 pieces +
  2 messages = 4 — and collapses to 2 when the history is empty.
- **`template | model` is a `RunnableSequence`.** An f-string cannot compose.

## Exercises

1. Add a fourth dropdown (say audience: student / engineer / manager). How many
   prompts now? At what point does "I can read them all" stop being true, and
   what would you do then?
2. Part B(c) showed the declaration being ignored. Find a template where that
   silent correction actually *hides* a bug from you, rather than helpfully
   fixing one.
3. Build the mislabelled history from Part C and the correct one, and send both
   to a real model with an API key. Does the answer actually differ? Design the
   question so that it must.
4. The Part D trap is silent. Write a three-line check you could put in a test
   suite that would catch it — before it reaches a paying API call.
5. `MessagesPlaceholder` inserts the whole history. Combine it with lesson 004's
   memory arithmetic: at what conversation length does this become the expensive
   thing, and which memory strategy would you reach for?
6. Rebuild the Part A template as an f-string and try to put it in a chain with
   `|`. What exactly is the error, and what does it tell you about what `|`
   requires?